# Survival Mixture Models for Government Processing Times
## Exponential-Weibull Mixtures with Censoring & Bayesian Updating

**Author:** Alexander Banoun  
**Stack:** statsmodels (custom MLE), SciPy, NumPy, Pandas  
**Data:** [NYC OpenData — License Applications](https://data.cityofnewyork.us/Business/License-Applications/ptev-4hud/about_data) (9,887 applications)

---

### Overview

How long does it take for a government agency to process a license application? The answer depends on the *type* of application — routine renewals may breeze through, while complex new filings sit in queue. Rather than fitting a single distribution, we model processing times as a **2-segment mixture**:

- **Segment 1 (Exponential):** Constant hazard rate — applications processed at a steady pace, typical of routine/automated workflows
- **Segment 2 (Weibull):** Time-varying hazard — applications requiring review, where processing probability changes with time in queue

The mixture probability depends on application characteristics (license type, application type, category), and applications still pending after 50 days are **right-censored**.

We then extend the analysis with a **multinomial logit** model predicting application *outcomes* (Issued, Denied, Withdrawn) and demonstrate **Bayesian updating** — computing how observing an approval on a specific day shifts our belief about which processing pathway an application followed.

---
## 1. Setup & Data

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as spst
import scipy.special as spsp
import matplotlib.pyplot as plt

from statsmodels.base.model import GenericLikelihoodModel
from statsmodels.tools.tools import add_constant
from statsmodels.formula.api import mnlogit

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

COLORS = {'blue': '#2563eb', 'orange': '#f59e0b', 'green': '#10b981', 
          'red': '#ef4444', 'purple': '#8b5cf6', 'gray': '#6b7280'}

In [ ]:
License = pd.read_csv("Archived_License_Applications.csv")
License = License.astype({"StartDate": "datetime64", "EndDate": "datetime64"})
License = License.assign(days=(License["EndDate"] - License["StartDate"]).dt.days + 1)
License = License[License["days"] >= 1]

# Right-censor at t=50 (applications still processing)
License.loc[License["days"] > 50, "days"] = 50
License["days"] = License["days"].fillna(50)

print(f"Applications: {len(License):,}")
print(f"Censored (≥50 days): {(License['days'] >= 50).sum():,} ({(License['days'] >= 50).mean():.1%})")
print(f"\nProcessing time distribution:")
print(License['days'].describe().to_string())

In [ ]:
# Visualize the processing time distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(License['days'], bins=50, color=COLORS['blue'], alpha=0.7, edgecolor='white')
ax1.axvline(x=50, color=COLORS['red'], linestyle='--', linewidth=2, label='Censoring threshold')
ax1.set_xlabel('Days to decision')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Processing Times')
ax1.legend()

status_counts = License['Status'].value_counts()
ax2.barh(status_counts.index, status_counts.values, color=[COLORS['green'], COLORS['red'], COLORS['orange'], COLORS['gray']])
ax2.set_xlabel('Count')
ax2.set_title('Application Outcomes')

for i, v in enumerate(status_counts.values):
    ax2.text(v + 50, i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 2. Exponential-Weibull Mixture Model

We fit a 2-segment mixture where the probability of belonging to each segment depends on three binary covariates:

| Covariate | Value = 1 | Value = 0 |
|-----------|-----------|-----------|
| LicenseType | Individual | Business |
| ApplicationType | Renewal | New Application |
| ApplicationCategory | Special | Basic |

The model has **7 parameters**: $\lambda_1$ (exponential rate), $c$ and $\lambda_2$ (Weibull shape/rate), and 4 logit coefficients ($\beta_0, \beta_1, \beta_2, \beta_3$) for the segment probability.

Because observations are discrete (integer days) and some are censored, we use **interval-censored likelihoods**: $P(T = t) = F(t) - F(t-1)$ for observed cases, and $P(T > 50) = 1 - F(50)$ for censored ones.

In [ ]:
class ExpoWeibull(GenericLikelihoodModel):
    """2-segment mixture: Exponential + Weibull with covariate-dependent mixing."""
    
    def loglike(self, params):
        y, X = self.endog, self.exog
        
        # Distribution parameters (log-transformed for positivity)
        lmbda1, c, lmbda2 = np.exp(params[0:3])
        
        # Censoring indicator
        censored = (y >= 50).astype(int)
        
        # Segment probability via logistic transformation
        p = 1 / (1 + np.exp(-(X @ params[3:])))
        
        # Segment 1: Exponential (Weibull with c=1)
        Et = spst.weibull_min.cdf(y, 1, scale=1/lmbda1)
        Et_m1 = spst.weibull_min.cdf(y - 1, 1, scale=1/lmbda1)
        
        # Segment 2: Weibull
        Wt = spst.weibull_min.cdf(y, c, scale=lmbda2**(-1/c))
        Wt_m1 = spst.weibull_min.cdf(y - 1, c, scale=lmbda2**(-1/c))
        
        # Likelihood: interval-censored for observed, survival for censored
        L1 = (1 - censored) * (Et - Et_m1) + censored * (1 - Et)
        L2 = (1 - censored) * (Wt - Wt_m1) + censored * (1 - Wt)
        
        return np.sum(np.log(p * L1 + (1 - p) * L2))

In [ ]:
# Encode covariates
License['LiType_Code'] = (License['LicenseType'] == 'Individual').astype(int)
License['App_Code'] = (License['ApplicationType'] == 'Renewal').astype(int)
License['AppCat_Code'] = (License['ApplicationCategory'] == 'Special').astype(int)

X = add_constant(License[['LiType_Code', 'App_Code', 'AppCat_Code']])

model = ExpoWeibull(License["days"], X)
result = model.fit(start_params=-np.random.random(7), method="bfgs")

print(result.summary(xname=[
    "log(λ₁)", "log(c)", "log(λ₂)", 
    "β₀ (intercept)", "β₁ (Individual)", "β₂ (Renewal)", "β₃ (Special)"
]))

In [ ]:
# Extract fitted parameters
lambda1, c, lambda2 = np.exp(result.params[0:3])
p0, p1, p2, p3 = result.params[3:]

print(f"\nFitted distribution parameters:")
print(f"  Segment 1 (Exponential): λ₁ = {lambda1:.4f}  →  mean = {1/lambda1:.1f} days")
print(f"  Segment 2 (Weibull):     c  = {c:.4f}, λ₂ = {lambda2:.4f}")
print(f"\nSegment probability coefficients:")
print(f"  β₀ = {p0:+.4f} (intercept)")
print(f"  β₁ = {p1:+.4f} (Individual license)")
print(f"  β₂ = {p2:+.4f} (Renewal)")
print(f"  β₃ = {p3:+.4f} (Special category)")

---
## 3. Hazard Functions & Segment Interpretation

The **hazard function** $h(t) = f(t) / S(t)$ gives the instantaneous probability of completion at time $t$, given survival to $t$. Comparing hazards reveals the qualitative difference between segments.

In [ ]:
t_int = np.arange(1, 21)
t_smooth = np.linspace(1, 20, 200)

# Interval-censored probabilities
Et = spst.weibull_min.cdf(t_int, 1, scale=1/lambda1)
Et_m1 = spst.weibull_min.cdf(t_int - 1, 1, scale=1/lambda1)
Wt = spst.weibull_min.cdf(t_int, c, scale=lambda2**(-1/c))
Wt_m1 = spst.weibull_min.cdf(t_int - 1, c, scale=lambda2**(-1/c))

# Continuous hazard functions
h_E = spst.weibull_min.pdf(t_smooth, 1, scale=1/lambda1) / spst.weibull_min.sf(t_smooth, 1, scale=1/lambda1)
h_W = spst.weibull_min.pdf(t_smooth, c, scale=lambda2**(-1/c)) / spst.weibull_min.sf(t_smooth, c, scale=lambda2**(-1/c))

# Integer hazard values for markers
h_E_int = spst.weibull_min.pdf(t_int, 1, scale=1/lambda1) / spst.weibull_min.sf(t_int, 1, scale=1/lambda1)
h_W_int = spst.weibull_min.pdf(t_int, c, scale=lambda2**(-1/c)) / spst.weibull_min.sf(t_int, c, scale=lambda2**(-1/c))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Interval-censored probabilities
ax1.plot(t_int, Et - Et_m1, 'o-', label='Seg 1: Exponential', color=COLORS['blue'], linewidth=2, markersize=6)
ax1.plot(t_int, Wt - Wt_m1, 'o-', label='Seg 2: Weibull', color=COLORS['orange'], linewidth=2, markersize=6)
ax1.set_xlabel('Days')
ax1.set_ylabel('P(decision on day t)')
ax1.set_title('Interval-Censored Decision Probabilities')
ax1.legend()

# Hazard functions
ax2.plot(t_smooth, h_E, color=COLORS['blue'], linewidth=2, label='Seg 1: Exponential')
ax2.plot(t_smooth, h_W, color=COLORS['orange'], linewidth=2, label='Seg 2: Weibull')
ax2.plot(t_int, h_E_int, 'o', color=COLORS['blue'], markersize=6)
ax2.plot(t_int, h_W_int, 'o', color=COLORS['orange'], markersize=6)
ax2.set_xlabel('Days')
ax2.set_ylabel('Hazard rate h(t)')
ax2.set_title('Hazard Functions by Segment')
ax2.legend()

plt.tight_layout()
plt.show()

**Interpretation:** Segment 1 (exponential) has a nearly **constant hazard** — the probability of a decision on any given day is roughly the same, regardless of how long the application has been waiting. This is consistent with routine, automated processing (renewals, simple licenses).

Segment 2 (Weibull with $c < 1$) has a **decreasing hazard** — the longer an application has waited, the *less* likely it is to be resolved soon. This suggests complex applications that may require additional review or get deprioritized over time.

---
## 4. Covariate Effects on Segment Membership

In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))

# Enumerate all feature combinations
combos = []
for lt in [0, 1]:
    for at in [0, 1]:
        for ac in [0, 1]:
            logit_val = p0 + p1*lt + p2*at + p3*ac
            prob = sigmoid(logit_val)
            combos.append({
                'LicenseType': 'Individual' if lt else 'Business',
                'ApplicationType': 'Renewal' if at else 'New',
                'Category': 'Special' if ac else 'Basic',
                'P(Seg 1)': prob,
                'P(Seg 2)': 1 - prob,
            })

df_combos = pd.DataFrame(combos).sort_values('P(Seg 1)', ascending=False)
print("Segment 1 probability by application profile:\n")
print(df_combos.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Visualize as horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 5))

labels = [f"{r['LicenseType']} / {r['ApplicationType']} / {r['Category']}" for _, r in df_combos.iterrows()]
probs = df_combos['P(Seg 1)'].values

bars = ax.barh(range(len(labels)), probs, color=COLORS['blue'], alpha=0.8)
ax.barh(range(len(labels)), 1 - probs, left=probs, color=COLORS['orange'], alpha=0.8)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Probability')
ax.set_title('Segment Membership by Application Profile')
ax.legend(['Seg 1 (fast/routine)', 'Seg 2 (slow/complex)'], loc='lower right')

for i, p in enumerate(probs):
    ax.text(p/2, i, f'{p:.1%}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

Renewals dominate Segment 1 (the fast track) — the $\beta_2$ coefficient (+2.02) is the strongest predictor. Special-category applications are pushed toward Segment 2 ($\beta_3 = -0.54$). The extreme case: an Individual Renewal for a Basic license has a **98%** chance of fast-track processing.

---
## 5. Bayesian Updating: Learning from Approval Timing

Suppose a **New, Basic, Business** license is approved on **day 7**. What does the timing tell us about which processing pathway it followed?

We start with a prior from the logistic model, then update using Bayes' rule:

$$P(\text{Seg 1} \mid \text{approved day 7}) = \frac{h_1(7) \cdot P(\text{Seg 1})}{h_1(7) \cdot P(\text{Seg 1}) + h_2(7) \cdot P(\text{Seg 2})}$$

In [ ]:
# Prior for New, Basic, Business application
logit_nbb = p0  # all covariates = 0
prior_seg1 = sigmoid(logit_nbb)

# Hazard at day 7
h1_day7 = spst.weibull_min.pdf(7, 1, scale=1/lambda1) / spst.weibull_min.sf(7, 1, scale=1/lambda1)
h2_day7 = spst.weibull_min.pdf(7, c, scale=lambda2**(-1/c)) / spst.weibull_min.sf(7, c, scale=lambda2**(-1/c))

# Bayesian update
p_day7 = h1_day7 * prior_seg1 + h2_day7 * (1 - prior_seg1)
posterior_seg1 = (h1_day7 * prior_seg1) / p_day7

print(f"Prior P(Seg 1 | New, Basic, Business):          {prior_seg1:.4f}")
print(f"Hazard at day 7:  Seg 1 = {h1_day7:.4f},  Seg 2 = {h2_day7:.4f}")
print(f"Posterior P(Seg 1 | approved on day 7):          {posterior_seg1:.4f}")
print(f"\nBelief shift: {prior_seg1:.1%} → {posterior_seg1:.1%}")

In [ ]:
# Visualize Bayesian updating across all days
days_range = np.arange(1, 51)
posteriors = []
for d in days_range:
    h1 = spst.weibull_min.pdf(d, 1, scale=1/lambda1) / spst.weibull_min.sf(d, 1, scale=1/lambda1)
    h2 = spst.weibull_min.pdf(d, c, scale=lambda2**(-1/c)) / spst.weibull_min.sf(d, c, scale=lambda2**(-1/c))
    denom = h1 * prior_seg1 + h2 * (1 - prior_seg1)
    posteriors.append((h1 * prior_seg1) / denom if denom > 0 else prior_seg1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(days_range, posteriors, color=COLORS['blue'], linewidth=2.5)
ax.axhline(y=prior_seg1, color=COLORS['gray'], linestyle='--', linewidth=1.5, label=f'Prior = {prior_seg1:.2f}')
ax.axvline(x=7, color=COLORS['red'], linestyle=':', linewidth=1.5, alpha=0.7, label='Day 7')
ax.scatter([7], [posterior_seg1], color=COLORS['red'], s=100, zorder=5)
ax.annotate(f'  {posterior_seg1:.2%}', (7, posterior_seg1), fontsize=11, color=COLORS['red'])

ax.set_xlabel('Day of approval')
ax.set_ylabel('P(Segment 1 | approved on day t)')
ax.set_title('Bayesian Posterior: How Approval Timing Reveals Processing Pathway')
ax.legend()
plt.tight_layout()
plt.show()

Early approvals strongly indicate Segment 1 (the routine pathway). An approval on day 7 updates our belief from **51.6%** to **91.7%** — the timing alone is highly informative. Later approvals shift the posterior back toward Segment 2, since the Weibull hazard dominates at longer durations.

---
## 6. Predicting Application Outcomes: Multinomial Logit

Beyond timing, we also model the *outcome* of each application using a multinomial logit with three categories: **Issued** (baseline), **Denied**, and **Withdrawn/Pending**.

In [ ]:
License['Status_numeric'] = License['Status'].map({
    'Issued': 0, 'Denied': 1, 'Withdrawn': 2, 'Pending': 2
})

mlogit = mnlogit("Status_numeric ~ LicenseType + ApplicationType + ApplicationCategory", data=License)
mlogit_result = mlogit.fit(method='newton', maxiter=100, disp=False)
print(mlogit_result.summary())

In [ ]:
# Compute probabilities for all feature combinations
params_deny = np.array([-3.0805, 0.6714, -1.5110, 0.6066])
params_other = np.array([-4.3637, 0.3512, -2.1708, 0.2669])

outcomes = []
for lt in [0, 1]:
    for at in [0, 1]:
        for ac in [0, 1]:
            x = np.array([1, lt, at, ac])
            logits = np.array([0, x @ params_deny, x @ params_other])
            probs = np.exp(logits) / np.exp(logits).sum()
            outcomes.append({
                'Profile': f"{'Indiv' if lt else 'Biz'} / {'Renew' if at else 'New'} / {'Special' if ac else 'Basic'}",
                'P(Issued)': probs[0], 'P(Denied)': probs[1], 'P(Other)': probs[2]
            })

df_out = pd.DataFrame(outcomes).sort_values('P(Denied)', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(df_out))
width = 0.55

p_issued = df_out['P(Issued)'].values
p_denied = df_out['P(Denied)'].values
p_other = df_out['P(Other)'].values

ax.bar(x_pos, p_issued, width, label='Issued', color=COLORS['green'], alpha=0.85)
ax.bar(x_pos, p_denied, width, bottom=p_issued, label='Denied', color=COLORS['red'], alpha=0.85)
ax.bar(x_pos, p_other, width, bottom=p_issued + p_denied, label='Withdrawn/Pending', color=COLORS['gray'], alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(df_out['Profile'].values, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Probability')
ax.set_title('Predicted Outcome Probabilities by Application Profile')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Renewals are almost always issued** — the $\beta_{\text{Renewal}}$ coefficient is strongly negative for both non-issuance outcomes (-1.51 for denial, -2.17 for withdrawal). **Individual + New + Special** applications face the highest denial risk, though even in the worst case, issuance probability exceeds 90%.

---
## 7. Summary

This project demonstrates three complementary modeling approaches for understanding government administrative processes:

**Survival mixture modeling** reveals that license applications follow two distinct processing pathways — a fast, constant-rate track (exponential) and a slower, decelerating track (Weibull). Application characteristics strongly predict which pathway an application enters.

**Bayesian updating** shows that observing *when* a decision is made provides powerful evidence about *how* it was processed. An early approval almost certainly came through the fast track, even for application types that are split 50/50 a priori.

**Multinomial logit** quantifies outcome probabilities, confirming that renewals are near-certain to be issued while new, individual, special-category applications face the highest (though still modest) denial risk.

```
  Application Features                    Timing Model                Outcome Model
  ┌──────────────────┐              ┌─────────────────────┐       ┌──────────────────┐
  │ License Type     │──────┐       │  Seg 1 (Exponential)│       │ P(Issued)        │
  │ Application Type │──────┼──────▶│  Seg 2 (Weibull)    │       │ P(Denied)        │
  │ Category         │──────┘       │  p(seg | features)  │       │ P(Withdrawn)     │
  └──────────────────┘              └────────┬────────────┘       └────────┬─────────┘
                                             │                            │
                                    Bayesian update                 Multinomial logit
                                    on observed timing              on same features
```

---
*Built as part of DDDM (Data-Driven Decision Making) coursework at Columbia University.*